# HMW station-hour 전체재학습 회귀앙상블 일괄실행 노트북

이 노트북은 **`Run All` 한 번으로** 학습/평가/결과 확인까지 진행할 수 있도록 정리한 실행 문서입니다.

## 노트북에서 하는 일
1. 입력 파일 존재 여부와 컬럼 스키마를 확인합니다.
2. 본 실험 스크립트를 실행해 회귀 앙상블 모델을 학습/평가합니다.
3. 모델 비교표, 잔차 요약, 오차 상위 샘플을 바로 확인합니다.
4. 요청이 많았던 대여소 `2359`, `3643`의 오차 샘플을 별도로 점검합니다.

## 입력 데이터
- 기본 경로: `3조 공유폴더/station_hour_bike_flow_2023_2025/station_hour_bike_flow_train_2023_2024.csv`
- 기본 경로: `3조 공유폴더/station_hour_bike_flow_2023_2025/station_hour_bike_flow_test_2025.csv`
- 선택 override: 환경변수 `DDRI_HMW_DATA_DIR`

## 산출물(스크립트 실행 후 생성)
- `회귀앙상블_모델비교_전체재학습.csv`
- `최적모델_잔차요약_전체재학습.csv`
- `최적모델_오차상위100_전체재학습.csv`
- `회귀앙상블_RMSE비교_전체재학습.png`
- `분석요약_전체재학습.md`
- `실험메타_전체재학습.json`


## 실행 전 체크리스트

- 이 노트북은 `ddri_work` 루트 혹은 그 하위 경로에서 실행하면 자동으로 경로를 찾도록 작성했습니다.
- 첫 실행은 데이터가 크기 때문에 시간이 걸릴 수 있습니다(환경에 따라 수 분~수십 분).
- 학습 중 커널을 중단하면 결과 파일이 일부만 생성될 수 있으니, 가능하면 끝까지 실행해 주세요.


In [ ]:
# 1) 공통 모듈 import + 경로 자동 탐색
# - 노트북 실행 위치가 달라도 동작하도록 루트 후보를 순회합니다.

from pathlib import Path
import json
import os
import time

import pandas as pd
from IPython.display import display, Markdown, Image

root_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
script_name = "run_hmw_full_station_hour_regression_ensemble.py"

script_path = None
for base in root_candidates:
    # 1순위: cheng80 하위 실험 폴더에서 스크립트 탐색
    cheng_dir = (base / "cheng80")
    if cheng_dir.exists():
        found = list(cheng_dir.rglob(script_name))
        if found:
            script_path = found[0].resolve()
            break

    # 2순위: 현재 기준 경로에서 직접 찾기
    candidate = (base / script_name).resolve()
    if candidate.exists():
        script_path = candidate
        break

# 노트북을 실험 폴더 안에서 직접 연 경우를 대비한 fallback
if script_path is None:
    found = list(Path.cwd().rglob(script_name))
    if found:
        script_path = found[0].resolve()

if script_path is None:
    raise FileNotFoundError("실험 스크립트를 찾지 못했습니다. ddri_work 루트 기준 경로를 확인해 주세요.")

work_dir = script_path.parent
root_dir = work_dir.parents[1]  # ddri_work
data_dir_candidates = []
if os.environ.get("DDRI_HMW_DATA_DIR"):
    data_dir_candidates.append(Path(os.environ["DDRI_HMW_DATA_DIR"]).expanduser())
data_dir_candidates.append(root_dir / "3조 공유폴더" / "station_hour_bike_flow_2023_2025")
data_dir_candidates.append(root_dir / "hmw" / "Data")

train_path = None
test_path = None
data_dir = None
for candidate_dir in data_dir_candidates:
    candidate_train = candidate_dir / "station_hour_bike_flow_train_2023_2024.csv"
    candidate_test = candidate_dir / "station_hour_bike_flow_test_2025.csv"
    if candidate_train.exists() and candidate_test.exists():
        data_dir = candidate_dir
        train_path = candidate_train
        test_path = candidate_test
        break

print(f"스크립트 경로: {script_path}")
print(f"작업 폴더: {work_dir}")
print(f"데이터 폴더: {data_dir}")
print(f"train 파일: {train_path.exists() if train_path else False} -> {train_path}")
print(f"test 파일: {test_path.exists() if test_path else False} -> {test_path}")

if train_path is None or test_path is None:
    searched = "\n".join(str(p) for p in data_dir_candidates)
    raise FileNotFoundError(f"필수 입력 파일(train/test)이 없습니다. 확인한 경로:\n{searched}")


In [ ]:
# 2) 입력 데이터 기본 점검
# - 열 이름, 행 개수, 샘플 데이터를 먼저 확인해 실행 전 오류를 줄입니다.

train_head = pd.read_csv(train_path, nrows=5)
test_head = pd.read_csv(test_path, nrows=5)

train_rows = sum(1 for _ in open(train_path, encoding="utf-8")) - 1
test_rows = sum(1 for _ in open(test_path, encoding="utf-8")) - 1

print(f"train 행 수: {train_rows:,}")
print(f"test 행 수: {test_rows:,}")
print("train 컬럼:", list(train_head.columns))
print("test 컬럼:", list(test_head.columns))

display(Markdown("### train 샘플(상위 5행)"))
display(train_head)
display(Markdown("### test 샘플(상위 5행)"))
display(test_head)


## 학습/평가 일괄 실행

아래 셀은 `run_hmw_full_station_hour_regression_ensemble.py`를 그대로 실행합니다.

- 내부에서 다중 회귀 모델(단일 + 앙상블)을 학습하고 성능을 비교합니다.
- `bike_change`를 타깃으로 RMSE/MAE/R²를 계산합니다.
- 최적 모델 기준 잔차 요약과 오차 상위 샘플을 저장합니다.
- 결과 CSV/PNG/JSON/MD 파일은 모두 현재 실험 폴더(`work_dir`)에 생성됩니다.


In [ ]:
# 3) 본 실험 스크립트 실행
# - subprocess + python -u로 실행해 모델별 로그를 실시간으로 확인합니다.

import subprocess
import sys

# Windows/로컬 환경에서 스레드 충돌 및 자원 과점을 줄이기 위한 안전 설정
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "1")

env = os.environ.copy()
env.setdefault("PYTHONUNBUFFERED", "1")

t0 = time.perf_counter()
process = subprocess.Popen(
    [sys.executable, "-u", str(script_path)],
    cwd=str(work_dir),
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in process.stdout:
    print(line, end="")

return_code = process.wait()
elapsed = time.perf_counter() - t0
print(f"총 실행시간: {elapsed:,.2f}초")
if return_code != 0:
    raise RuntimeError(f"실험 스크립트 실행 실패: return_code={return_code}")


## 결과 해석 가이드

- `RMSE`, `MAE`는 **낮을수록 좋습니다**.
- `R²`는 **높을수록 좋습니다**(일반적으로 1에 가까울수록 설명력이 높음).
- 모델 선택은 보통 `RMSE` 우선, 동률/근접 시 `MAE`, `R²`, 학습시간을 함께 봅니다.


In [ ]:
# 4) 모델 비교표 확인 + 상위 모델 확인

model_cmp_path = work_dir / "회귀앙상블_모델비교_전체재학습.csv"
model_cmp = pd.read_csv(model_cmp_path, encoding="utf-8-sig")

# 실수로 정렬이 바뀐 경우를 대비해 RMSE 기준으로 다시 정렬
model_cmp = model_cmp.sort_values(["RMSE", "MAE"], ascending=[True, True]).reset_index(drop=True)

display(Markdown("### 회귀앙상블 모델 성능 비교"))
display(model_cmp)

best_row = model_cmp.iloc[0]
display(Markdown(
    f"**최적 모델(현재 기준)**: `{best_row['모델']}`  \n"
    f"- RMSE: `{best_row['RMSE']:.6f}`  \n"
    f"- MAE: `{best_row['MAE']:.6f}`  \n"
    f"- R²: `{best_row['R2']:.6f}`"
))


In [ ]:
# 5) RMSE 비교 그래프 확인

img_path = work_dir / "회귀앙상블_RMSE비교_전체재학습.png"
if img_path.exists():
    display(Markdown("### 모델별 RMSE 비교 그래프"))
    display(Image(filename=str(img_path)))
else:
    print(f"그래프 파일이 없습니다: {img_path}")


In [ ]:
# 6) 최적 모델 잔차 요약 확인

residual_path = work_dir / "최적모델_잔차요약_전체재학습.csv"
residual_summary = pd.read_csv(residual_path, encoding="utf-8-sig")
display(Markdown("### 최적 모델 잔차 요약"))
display(residual_summary)


In [ ]:
# 7) 오차 상위 샘플 + 특정 대여소(2359, 3643) 점검
# - 전체 상위 오차를 보고, 요청이 있었던 대여소만 별도로 필터링합니다.

top_err_path = work_dir / "최적모델_오차상위100_전체재학습.csv"
top_err = pd.read_csv(top_err_path, encoding="utf-8-sig")

display(Markdown("### 오차 상위 샘플(상위 20행)"))
display(top_err.head(20))

if "station_id" in top_err.columns:
    station_focus = top_err[top_err["station_id"].astype(str).isin(["2359", "3643"])].copy()
    display(Markdown("### 대여소 2359 / 3643 오차 샘플"))
    if len(station_focus) == 0:
        display(Markdown("상위 오차 100건 안에는 2359/3643 데이터가 없습니다."))
    else:
        display(station_focus)
else:
    print("주의: 오차 파일에 station_id 컬럼이 없어 2359/3643 필터를 건너뜁니다.")


In [ ]:
# 8) 실험 메타/요약 파일 확인 + 생성 파일 목록 점검

meta_path = work_dir / "실험메타_전체재학습.json"
summary_md_path = work_dir / "분석요약_전체재학습.md"

meta = json.loads(meta_path.read_text(encoding="utf-8"))
display(Markdown("### 실험 메타 정보"))
display(meta)

display(Markdown("### 요약 마크다운 파일 미리보기(앞 30줄)"))
summary_lines = summary_md_path.read_text(encoding="utf-8").splitlines()[:30]
print("\n".join(summary_lines))

display(Markdown("### 생성 파일 목록"))
for p in sorted(work_dir.glob("*")):
    if p.is_file():
        print(f"- {p.name}")


## 트러블슈팅

### 1) `FileNotFoundError`가 발생할 때
- 노트북의 1번 셀에서 출력한 `script_path`, `train_path`, `test_path`를 먼저 확인하세요.
- 파일명이 비슷해도 연도(train 2023~2024 / test 2025)가 다르면 결과가 달라집니다.

### 2) 실행 시간이 너무 길 때
- `run_hmw_full_station_hour_regression_ensemble.py`의 `model_configs()`에서 샘플 cap을 낮추면 빨라집니다.
- 먼저 `랜덤포레스트`, `엑스트라트리`, `그래디언트부스팅` 위주로 빠르게 비교한 뒤 확장하는 것을 권장합니다.

### 3) 결과가 기대와 다를 때
- `최적모델_오차상위100_전체재학습.csv`에서 특정 시간대/대여소에 오차가 집중되는지 확인하세요.
- 필요하면 대여소별, 시간대별 서브모델로 분리해 재학습하는 것이 효과적입니다.
